# Tutorial 1: Hardware-aware neural architecture search on MNIST

This notebook runs the complete SNAC-Pack pipeline end to end on the MNIST
handwritten-digit dataset.

| Step | Description |
|---|---|
| 1 | Load and visualize the dataset |
| 2 | Run a hardware-aware global search using NSGA-II and the `rule4ml` surrogate |
| 3 | Examine the Pareto front: accuracy versus chip cost |
| 4 | Compress the best architecture with QAT and iterative magnitude pruning |
| 5 | (Extension) Search over Conv and ConvAttn block types |

All search parameters are controlled by `t1_config.yaml` in this directory.
The slides cover the motivation and theory behind each stage; this notebook
implements the pipeline.

In [ ]:
!which python

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent.parent          # repo root
sys.path.insert(0, str(ROOT))

import yaml
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from utils.tf_global_search import GlobalSearchTF
from utils.tf_local_search_separated import local_search_entrypoint
from utils.tf_data_preprocessing import load_and_preprocess_mnist
from utils.tf_visualization import plot_pareto_fronts, plot_interactive_2d_pareto

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
tf.get_logger().setLevel('ERROR')
print('TensorFlow:', tf.__version__)

cfg     = yaml.safe_load(open(Path.cwd() / 't1_config.yaml'))
ds_cfg  = cfg['dataset']
s_cfg   = cfg['search']
ss_cfg  = cfg['search_space']
ls_cfg  = cfg['local_search']
out_cfg = cfg['output']

RESULTS_DIR = out_cfg['results_dir']
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir  : {RESULTS_DIR}')
print(f'n_trials={s_cfg["n_trials"]}, epochs={s_cfg["epochs"]}, hw_metrics={s_cfg["use_hardware_metrics"]}')

## The codesign premise

Every architectural choice (layer count, width, activation function, quantization
precision) affects accuracy, chip area (LUTs, DSPs, BRAM), and inference latency
simultaneously (slide 3).
Traditional NAS optimizes accuracy and evaluates hardware cost only at the end.
SNAC-Pack incorporates hardware cost directly as a search objective, producing a
Pareto set of architectures rather than a single answer.

<details>
<summary>Background: machine learning</summary>

A neural network is a parameterised function composed of layers of multiply-accumulate
operations. Training adjusts the weight parameters so that the network maps each input
(e.g. a digit image) to the correct output label (0-9).
The term *architecture* refers to the network's shape: depth, layer widths, and
activation functions.

</details>

<details>
<summary>Background: FPGAs and resource metrics</summary>

An FPGA (Field-Programmable Gate Array) is a chip whose logic fabric is configured at
run time. Unlike a GPU, it has no fixed multiply-accumulate units; arithmetic is
synthesised from LUTs (look-up tables) and DSPs (fast multiplier blocks).
Smaller networks consume fewer LUTs and DSPs, allowing deployment on smaller and
lower-power devices.
hls4ml translates a trained Keras model into HLS C++, which Vivado then synthesises to
an FPGA bitstream.

</details>

**Note on rule4ml**: hardware metric estimation requires the conda library path.
If the global search fails with a shared-library error, set the following in your
terminal before launching Jupyter:
```
export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:${LD_LIBRARY_PATH:-}"
```

## Dataset: MNIST

Each sample is a greyscale image of a handwritten digit (0-9), resized to 8x8
pixels and flattened to a 64-dimensional vector for MLP input.

In [ ]:
x_viz, y_viz, _, _ = load_and_preprocess_mnist(
    resize_val=ds_cfg['resize_val'],
    subset_size=200,
    flatten=False,
    one_hot=False,
)

plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_viz[i].squeeze(), cmap='gray')
    plt.title(f'Label: {y_viz[i]}')
    plt.axis('off')
plt.suptitle(f'MNIST digits resized to {ds_cfg["resize_val"]}x{ds_cfg["resize_val"]}')
plt.tight_layout()
plt.show()

## The configuration file

SNAC-Pack is entirely config-driven (slide 17).
The table below maps each YAML block to the corresponding slide.

| Block | Controls | Slide |
|---|---|---|
| `dataset` | Task, resize, subset size | 17 |
| `search` | Trial budget, objectives, surrogate on/off | 8, 14 |
| `search_space` | Layer types, widths, activations | 18 |
| `hls_config` | Target FPGA board, precision, reuse factor | 16-17 |
| `local_search` | QAT precision pairs, pruning schedule | 9-11 |

The cell below prints the file you are about to run.

In [ ]:
with open(Path.cwd() / 't1_config.yaml') as fh:
    print(fh.read())

## Fixed-point quantization (slide 10)

On FPGAs, weights are represented as fixed-point numbers `ap_fixed<W, I>`,
where W is the total bit-width and I is the number of integer bits.
Reducing W shrinks the multipliers and lowers resource cost;
too few bits introduces rounding error and degrades accuracy.
Quantization-aware training (QAT) recovers accuracy by simulating the rounding
during the training forward pass.

The cell below shows how a single weight value is approximated at several precisions.
Use the slider to vary the maximum bit-width displayed.

In [ ]:
_PRECISIONS = [
    ('FP32',           32, 0.72341),
    ('ap_fixed<16,6>', 16, 0.72363),
    ('ap_fixed<8,3>',   8, 0.71875),
    ('ap_fixed<6,2>',   6, 0.71875),
    ('ap_fixed<4,1>',   4, 0.75000),
]

def _show_quant_table(max_bits):
    fp32_val = _PRECISIONS[0][2]
    header = f'{"Format":<20}{"Bits":>6}{"Value":>12}{"Rounding error":>18}'
    print(header)
    print('-' * len(header))
    for name, bits, val in _PRECISIONS:
        if bits <= max_bits:
            err = abs(val - fp32_val)
            print(f'{name:<20}{bits:>6}{val:>12.5f}{err:>18.5f}')

try:
    import ipywidgets as widgets
    widgets.interact(
        _show_quant_table,
        max_bits=widgets.SelectionSlider(
            options=[4, 6, 8, 16, 32],
            value=32,
            description='Max bits:',
            style={'description_width': 'initial'},
        ),
    )
except ImportError:
    _show_quant_table(32)

## Stage 1: Global search with hardware surrogate

NSGA-II samples candidate architectures, trains each briefly, then calls `rule4ml`
to estimate LUT%, DSP%, BRAM%, and clock cycles without running Vivado
(milliseconds per trial rather than hours).
Non-dominated solutions advance to the next generation; the final output is a
Pareto front.

<details>
<summary>Background: avg_resource and clock_cycles</summary>

`avg_resource` is the arithmetic mean of the LUT%, DSP%, BRAM%, and FF% estimates
returned by `rule4ml`. A lower value indicates a more resource-efficient model.
`clock_cycles` is the predicted inference latency in clock cycles at the configured
clock frequency. At 250 MHz, 10 cycles corresponds to approximately 40 ns.

</details>

In [ ]:
obj_names = s_cfg['objective_names']
max_flags = s_cfg['maximize_flags']

searcher = GlobalSearchTF(
    search_space_path=ss_cfg,
    results_dir=RESULTS_DIR,
)

study = searcher.run_search(
    model_type=s_cfg['model_type'],
    n_trials=s_cfg['n_trials'],
    epochs=s_cfg['epochs'],
    dataset=ds_cfg['name'],
    subset_size=ds_cfg['subset_size'],
    resize_val=ds_cfg['resize_val'],
    objectives=obj_names,
    maximize_flags=max_flags,
    use_hardware_metrics=s_cfg['use_hardware_metrics'],
)

print('Global search complete.')

## Pareto front analysis

The search returns a set of Pareto-optimal architectures, not a single best model
(slide 5/15).
Each point on the front represents a genuine trade-off; the choice of which model
to use depends on the application's resource and latency budget.
Hovering over a point in the interactive plot displays the full architecture
description for that trial.

In [ ]:
results_df = pd.DataFrame(searcher.results)

if not results_df.empty:
    best = results_df.loc[results_df['performance_metric'].idxmax()]
    print(f'Highest-accuracy trial : {int(best["trial"])}  '
          f'Accuracy : {best["performance_metric"]:.4f}  '
          f'BOPs : {best["bops"]:.2e}')
    print(f'Pareto plots saved to  : {RESULTS_DIR}')

    obj_info = list(zip(obj_names, max_flags))
    plot_pareto_fronts(results_df, obj_info, save_dir=RESULTS_DIR, show=True)
    plot_interactive_2d_pareto(results_df, obj_info, save_dir=RESULTS_DIR, show=True)
else:
    print('No results available. Run the global search cell above first.')

## Exercise: modify the search and observe the effect

Before running the cell below, consider the following question:

Does a wide, shallow network (e.g. two layers of 128 neurons) consume more or fewer
LUTs than a narrow, deep network (e.g. six layers of 16 neurons)?
Modify the parameters, re-run the global search cell, and examine how the Pareto
front shifts.

This exercise illustrates the central principle: **SNAC-Pack is controlled entirely
through the YAML configuration**.

In [ ]:
# Modify these values, then re-run the 'Stage 1: Global search' cell above.

cfg['search']['n_trials'] = 5                          # suggested range: 3-10
cfg['search_space']['mlp_width_space'] = [8, 32, 128]  # try [8, 16] or [64, 128, 256]

# To disable the hardware surrogate and optimize accuracy alone, uncomment:
# cfg['search']['use_hardware_metrics'] = False

# Refresh shorthand references so GlobalSearchTF picks up the changes.
s_cfg     = cfg['search']
ss_cfg    = cfg['search_space']
obj_names = s_cfg['objective_names']
max_flags = s_cfg['maximize_flags']

print('Configuration updated.')
print(f'  n_trials   : {s_cfg["n_trials"]}')
print(f'  widths     : {ss_cfg["mlp_width_space"]}')
print(f'  hw_metrics : {s_cfg["use_hardware_metrics"]}')
print()
print('Re-run the "Stage 1: Global search" cell to apply.')

## Stage 2: Local search: quantization and pruning

The global search determines the architecture shape.
Local search compresses the selected model using two techniques.

**Quantization-aware training (QAT)** retrains the model while simulating
fixed-point rounding in the forward pass, so the network learns to tolerate the
precision loss. The search sweeps all `precision_pairs` specified in the config
(slides 10, 11).

**Iterative magnitude pruning** ranks weights by absolute value, zeros the smallest
fraction, and retrains to recover accuracy. This is repeated for several iterations.
Typical results: 70-95% sparsity with less than 1% accuracy degradation (slide 11).

In [ ]:
# Reload the original config to ensure local search parameters are unaffected
# by any modifications made in the Exercise cell above.
cfg_orig    = yaml.safe_load(open(Path.cwd() / 't1_config.yaml'))
ds_cfg_orig = cfg_orig['dataset']
ls_cfg_orig = cfg_orig['local_search']
RESULTS_DIR = cfg_orig['output']['results_dir']

LOCAL_RESULTS_DIR = os.path.join(RESULTS_DIR, 'local_search_separated')
LOCAL_CONFIG_PATH = os.path.join(RESULTS_DIR, 'local_search_config.yaml')
ARCH_YAML_PATH    = os.path.join(RESULTS_DIR, 'best_model_for_local_search.yaml')

local_search_settings = {
    'pruning_settings': {
        'iterations':           ls_cfg_orig['pruning_iterations'],
        'epochs_per_iteration': ls_cfg_orig['pruning_epochs'],
        'pruning_rate':         ls_cfg_orig['pruning_rate'],
    },
    'qat_settings': {
        'epochs':          ls_cfg_orig['qat_epochs'],
        'precision_pairs': ls_cfg_orig['precision_pairs'],
    },
}
with open(LOCAL_CONFIG_PATH, 'w') as fh:
    yaml.dump(local_search_settings, fh)

x_train, y_train, x_val, y_val = load_and_preprocess_mnist(
    resize_val=ds_cfg_orig['resize_val'],
    subset_size=ds_cfg_orig['subset_size'],
    flatten=True,
    one_hot=True,
)

if os.path.exists(ARCH_YAML_PATH):
    pruning_df, qat_df = local_search_entrypoint(
        architecture_yaml_path=ARCH_YAML_PATH,
        local_search_config_path=LOCAL_CONFIG_PATH,
        dataset=(x_train, y_train, x_val, y_val),
        results_dir=LOCAL_RESULTS_DIR,
    )
else:
    print(f'Architecture YAML not found: {ARCH_YAML_PATH}')
    print('Complete the global search first, then return to this cell.')
    pruning_df, qat_df = pd.DataFrame(), pd.DataFrame()

In [ ]:
if not pruning_df.empty:
    plt.figure(figsize=(10, 4))
    plt.plot(pruning_df['Sparsity'], pruning_df['Accuracy'],
             marker='o', linewidth=2, color='teal')
    plt.title('Pruning: accuracy vs. sparsity')
    plt.xlabel('Sparsity (fraction of weights set to zero)')
    plt.ylabel('Accuracy')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

if not qat_df.empty:
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(qat_df)), qat_df['Accuracy'],
            tick_label=qat_df['Precision'], color='teal')
    plt.title('QAT: accuracy vs. fixed-point precision')
    plt.xlabel('Precision (ap_fixed<W,I>)')
    plt.ylabel('Accuracy')
    plt.tight_layout()
    plt.show()

## Pruning visualization

The slider below shows how the weight matrix changes as the sparsity level increases.
Active weights are shown as filled cells; pruned weights are blank.
The right panel marks the current sparsity level on the accuracy curve from the local
search above.

In [ ]:
_rng = np.random.default_rng(42)
_W   = _rng.random((10, 10))

_spar_vals = pruning_df['Sparsity'].tolist() if not pruning_df.empty else []
_acc_vals  = pruning_df['Accuracy'].tolist() if not pruning_df.empty else []

def _show_pruning(sparsity_pct):
    sparsity = sparsity_pct / 100.0
    mask     = _rng.random((10, 10)) > sparsity
    sparse_W = _W * mask
    n_active = int(mask.sum())

    ncols = 3 if _spar_vals else 2
    fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 3.5))

    axes[0].imshow(_W > 0, cmap='Blues', vmin=0, vmax=1)
    axes[0].set_title('Dense  (100 weights active)')
    axes[0].axis('off')

    axes[1].imshow(sparse_W != 0, cmap='Blues', vmin=0, vmax=1)
    axes[1].set_title(f'Sparse  ({n_active}/100 active)')
    axes[1].axis('off')

    if _spar_vals:
        axes[2].plot(_spar_vals, _acc_vals, 'o-', color='teal', linewidth=2)
        axes[2].axvline(x=sparsity, color='red', linestyle='--',
                        label=f'{sparsity_pct}% zeroed')
        axes[2].set(xlabel='Sparsity', ylabel='Accuracy',
                    title='Accuracy vs. sparsity')
        axes[2].legend()
        axes[2].grid(True, linestyle='--', alpha=0.5)

    fig.suptitle(f'Sparsity = {sparsity_pct}%', fontsize=13)
    plt.tight_layout()
    plt.show()

try:
    import ipywidgets as widgets
    widgets.interact(
        _show_pruning,
        sparsity_pct=widgets.IntSlider(
            min=0, max=90, step=10, value=0,
            description='Sparsity %:',
            style={'description_width': 'initial'},
        ),
    )
except ImportError:
    _show_pruning(0)

---
## Extension: searching over convolutional and attention blocks

The search space supports four block types: **Conv**, **ConvAttn**, **MLP**, and
**None** (identity skip). Modern architectures including self-attention blocks are
therefore fully searchable.

**Why is the hardware surrogate disabled for this extension?**
`rule4ml` estimates resources only for networks with ReLU, Softmax, and linear
activations; it also flattens models before evaluation, which removes the spatial
structure that convolutional and attention blocks require.
With `use_hardware_metrics: false`, SNAC-Pack searches the full block space but
optimizes accuracy alone, hardware cost is not included in the search objective.

This is an explicit trade-off: one toggle (`use_hardware_metrics`) and one
`block_types` entry separate hardware-aware codesign from unconstrained architecture
exploration.

This cell loads `t1_conv_config.yaml`, which configures a Conv/ConvAttn/MLP/None
block search on MNIST with the hardware surrogate disabled.

In [ ]:
conv_cfg     = yaml.safe_load(open(Path.cwd() / 't1_conv_config.yaml'))
conv_ds_cfg  = conv_cfg['dataset']
conv_s_cfg   = conv_cfg['search']
conv_ss_cfg  = conv_cfg['search_space']
conv_out_cfg = conv_cfg['output']

CONV_RESULTS_DIR = conv_out_cfg['results_dir']
os.makedirs(CONV_RESULTS_DIR, exist_ok=True)

conv_obj_names = conv_s_cfg['objective_names']
conv_max_flags = conv_s_cfg['maximize_flags']

print(f'Block types       : {conv_ss_cfg["block_types"]}')
print(f'Hardware metrics  : {conv_s_cfg["use_hardware_metrics"]}  (surrogate disabled)')
print(f'n_trials          : {conv_s_cfg["n_trials"]}')
print()

conv_searcher = GlobalSearchTF(
    search_space_path=conv_ss_cfg,
    results_dir=CONV_RESULTS_DIR,
)

conv_study = conv_searcher.run_search(
    model_type=conv_s_cfg['model_type'],
    n_trials=conv_s_cfg['n_trials'],
    epochs=conv_s_cfg['epochs'],
    dataset=conv_ds_cfg['name'],
    subset_size=conv_ds_cfg['subset_size'],
    resize_val=conv_ds_cfg['resize_val'],
    objectives=conv_obj_names,
    maximize_flags=conv_max_flags,
    use_hardware_metrics=conv_s_cfg['use_hardware_metrics'],
    n_folds=conv_s_cfg.get('n_folds', 1),
)
print('Block search complete.')

In [ ]:
conv_results_df = pd.DataFrame(conv_searcher.results)
if not conv_results_df.empty:
    best_conv = conv_results_df.loc[conv_results_df['performance_metric'].idxmax()]
    print(f'Best trial   : {int(best_conv["trial"])}')
    print(f'Accuracy     : {best_conv["performance_metric"]:.4f}')
    print(f'BOPs         : {best_conv["bops"]:.2e}')
    print()

    block_cols = [c for c in conv_results_df.columns if 'block' in c and 'type' in c]
    if block_cols:
        top = conv_results_df.nlargest(min(5, len(conv_results_df)), 'performance_metric')
        print('Top trials and their selected block types:')
        print(top[['trial', 'performance_metric', 'bops'] + block_cols].to_string(index=False))
        print()

    conv_obj_info = list(zip(conv_obj_names, conv_max_flags))
    plot_pareto_fronts(conv_results_df, conv_obj_info, save_dir=CONV_RESULTS_DIR, show=True)
else:
    print('No results available. Run the block search cell above first.')

## Summary

| Stage | Description |
|---|---|
| Global search (surrogate enabled) | NSGA-II over MLP architectures; `rule4ml` estimates chip cost per trial |
| Pareto front | Set of accuracy-resource trade-offs; model selection depends on application budget |
| Local search | Best architecture compressed with QAT and iterative magnitude pruning |
| Extension (surrogate disabled) | Same pipeline applied to Conv/ConvAttn block space |

Tutorial 2 applies this pipeline to qubit readout data with an explicit latency
constraint, and concludes with hls4ml synthesis to FPGA hardware description code.